# 03. Schema drift and time travel

Adds a new column to the table, appends a new batch, and shows that old rows read NULL and old snapshots still work.

## What to look at

- The **schema diff**: only `review_channel` is new.
- **Old rows show NULL** for it; the new batch has app/web/marketplace/email.
- **Time travel**: the first snapshot still returns the old row count with the *old* schema.

In [ ]:
import sys, pathlib
_nb = next(p for p in (pathlib.Path.cwd(), pathlib.Path.cwd() / "notebooks") if (p / "_setup.py").exists())
sys.path.insert(0, str(_nb))
from _setup import *   # puts src/ on the path, loads .env, gives run_script / get_spark / sql / minio helpers
print("SAMPLE_ROWS =", common.SAMPLE_ROWS, "(None = all rows)")

In [ ]:
# re-run safety: if drift was already applied, rebuild bronze first so the demo starts clean
spark = get_spark("nb03")
if "review_channel" in spark.table(T_BRONZE).columns:
    print("drift already applied -> re-running 02_bronze_ingest to reset")
    run_script("02_bronze_ingest.py", show_log=False)
    spark = get_spark("nb03")
cols_before = spark.table(T_BRONZE).columns
rows_before = spark.table(T_BRONZE).count()
print(len(cols_before), "columns,", rows_before, "rows before")

In [ ]:
run_script("03_schema_drift.py")

In [ ]:
spark = get_spark("nb03")
cur = spark.table(T_BRONZE)
pd.DataFrame({"added columns": [c for c in cur.columns if c not in cols_before]})

In [ ]:
sql(spark, f"""SELECT coalesce(review_channel, '(NULL: old rows)') AS review_channel, count(*) AS rows
               FROM {T_BRONZE} GROUP BY 1 ORDER BY 2 DESC""")

In [ ]:
snaps = sql(spark, f"SELECT committed_at, snapshot_id, parent_id, operation FROM {T_BRONZE}.snapshots ORDER BY committed_at")
snaps

In [ ]:
first = int(snaps.snapshot_id.iloc[0])
old = spark.sql(f"SELECT * FROM {T_BRONZE} VERSION AS OF {first}")
pd.DataFrame([("current table", cur.count(), "review_channel" in cur.columns),
              (f"VERSION AS OF {first}", old.count(), "review_channel" in old.columns)],
             columns=["what you read", "rows", "has review_channel column?"])

In [ ]:
# Cleanup. Skip this cell while you are showing the Spark UI (http://localhost:4040): it closes with the session.
stop_spark()

## Open MinIO now

http://localhost:9001 (login `minioadmin` / `minioadmin`), bucket **warehouse**.

Go into `bronze/reviews/metadata/`. There are now **more `*.metadata.json` files** (one for the ADD COLUMN, one for the append) and a **second `snap-*.avro` manifest list**. Open `data/` again: the old Parquet files are unchanged and a few new ones were added for the new batch. ADD COLUMN rewrote no data.